# 03 - Train the core models (proposed, baselines, seeds)

**Settings:** Accelerator = **GPU T4 x2**, Internet = On. Run with **Save Version > Save & Run All (Commit)**,
not the editor's Run All: a committed run keeps going when the laptop is closed and its output is saved.

**Inputs** (Add Input > Your Work > Notebooks):
1. `01_prepare_trainval` (ufm-trainval)
2. `02_prepare_test` (ufm-test)

**Output:** `runs/<experiment>/` (checkpoints, metrics, maps)  |  **Approx. time:** 4-6 h

Proposed model D, the dataset authors' setup B5, the 8-band model A, and two extra random seeds of A and D (mean +- sd). D also gets 20-pass MC-dropout uncertainty maps.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import glob, os, subprocess
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_input(name, notebook):
    """Locate an attached input folder or stop immediately with a clear message."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise SystemExit(f"STOP: input '{name}' not found. Add the output of notebook {notebook} "
                         "(Add Input > Your Work > Notebooks) and run again.")
    print(f"{name}: {hits[0]}")
    return hits[0]

In [ ]:
TRAINVAL = find_input("ufm-trainval", "01_prepare_trainval")
TEST = find_input("ufm-test", "02_prepare_test")
JOBS = ['D_full', 'B5_authors_baseline', 'A_raw', 'D_full@1', 'A_raw@1', 'D_full@2', 'A_raw@2']      # "<experiment>@<k>" = extra random seed k
MC = ['D_full']

In [ ]:
!pip install -q segmentation-models-pytorch tabulate rasterio 2>/dev/null; python -c "import rasterio, segmentation_models_pytorch as s; print('ok', s.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# trains two experiments at a time (one per T4); progress lines are prefixed [gpu0]/[gpu1]
!python scripts/run_queue.py {' '.join(JOBS)} --trainval-root {TRAINVAL} --test-root {TEST} --out /kaggle/working/runs --mc {' '.join(MC) or '""'}

In [ ]:
!for f in /kaggle/working/runs/*/test/test_metrics.csv; do echo "== $f"; grep ALL $f | cut -c1-120; done